# Лабораторна робота 11

In [1]:
# %% [Р1.1] Завантаження корпусу й очищення тексту (колонтитули, пробіли)
import sys, json, re, time, statistics as st
sys.path.insert(0, "..")
import warnings, logging; warnings.filterwarnings("ignore"); logging.getLogger("transformers").setLevel(logging.ERROR)
from transformers.utils import logging as hf_logging; hf_logging.set_verbosity_error()
from pathlib import Path
import numpy as np, pandas as pd
from pcdiag import rag
from pcdiag.rag import load_docs, chunk_fixed, chunk_structural, DenseIndex, BM25Index, HybridRetriever, rrf

pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 80)
docs = load_docs("corpus")
print(pd.DataFrame([{k: d[k] for k in ("doc", "title", "lang", "removed", "raw_chars", "clean_chars")} for d in docs])
      .to_string(index=False))
print(f"\nДокументів: {len(docs)}; видалено рядків-колонтитулів: {sum(d['removed'] for d in docs)}; "
      f"символів до/після: {sum(d['raw_chars'] for d in docs)} / {sum(d['clean_chars'] for d in docs)}")
print("Ліцензія корпусу: власні тексти автора (CC BY 4.0), створені для ЛР; дані про обладнання — демонстраційні.")
raw = (Path("corpus") / docs[0]["file"]).read_text(encoding="utf-8").splitlines()
print("\nДо очищення (KB-01, рядки 11–14):", *raw[10:14], sep="\n  ")
print("Після очищення той самий фрагмент:", *[l for _, l in docs[0]["lines"][8:10]], sep="\n  ")

  doc                                             title lang  removed  raw_chars  clean_chars
KB-01                                Перегрів процесора   uk        4       1975         1811
KB-02                               Відмова вентилятора   uk        4       1530         1359
KB-03                        Діагностика блока живлення   uk        4       1689         1523
KB-04                                    Батарейка CMOS   uk        2       1161         1080
KB-05                    Тестування оперативної пам'яті   uk        4       1644         1476
KB-06                 XMP і нестабільний розгін пам'яті   uk        2       1269         1191
KB-07                              Звукові сигнали POST   uk        4       1425         1257
KB-08                        Коди синього екрана (BSOD)   uk        4       1674         1508
KB-09                       S.M.A.R.T. і збійні сектори   uk        4       1825         1664
KB-10        Заміна накопичувача та резервне копіювання   uk

In [2]:
# %% [Р1.2] Дві стратегії chunking і метадані
F = chunk_fixed(docs, size=400, overlap=0.15)
S = chunk_structural(docs, max_tokens=200)
def stats(ch):
    t = [c["tokens"] for c in ch]
    return {"фрагментів": len(ch), "сер. токенів": round(st.mean(t), 1), "мін": min(t), "макс": max(t),
            "> 128 токенів (вікно моделі)": sum(x > 128 for x in t), "на документ": round(len(ch) / len(docs), 2)}
print(pd.DataFrame({"фіксований 400 / 15 %": stats(F), "структурний (розділ ##)": stats(S)}).to_string())
print("\nМетадані фрагмента (структурний):"); print({k: v for k, v in S[15].items() if k != "text"})
print("Метадані фрагмента (фіксований):"); print({k: v for k, v in F[5].items() if k != "text"})
Path("index").mkdir(exist_ok=True)
for name, ch in [("fixed", F), ("structural", S)]:
    with open(f"index/chunks_{name}.jsonl", "w", encoding="utf-8") as f:
        for c in ch: f.write(json.dumps(c, ensure_ascii=False) + "\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

                              фіксований 400 / 15 %  структурний (розділ ##)
фрагментів                                    26.00                    69.00
сер. токенів                                 276.30                   114.60
мін                                           99.00                    58.00
макс                                         400.00                   207.00
> 128 токенів (вікно моделі)                  24.00                    21.00
на документ                                    1.18                     3.14

Метадані фрагмента (структурний):
{'id': 'KB-05:s1', 'doc': 'KB-05', 'title': "Тестування оперативної пам'яті", 'section': 'Засоби', 'page': 1, 'lang': 'uk', 'strategy': 'structural', 'tokens': 111}
Метадані фрагмента (фіксований):
{'id': 'KB-04:f0', 'doc': 'KB-04', 'title': 'Батарейка CMOS', 'section': 'Призначення', 'page': 1, 'lang': 'uk', 'strategy': 'fixed', 'tokens': 275}


In [3]:
# %% [Р1.3] Порівняння стратегій на прикладах «поганих» розрізів
def show(c, n=160):
    return f"[{c['id']}] стор.{c['page']} «{c['section']}» ({c['tokens']} ток.): …{c['text'][-n:]}"
# фіксований фрагмент, що обривається посеред нумерованого списку / речення
bad = [c for c in F if not re.search(r"[.!?:»)]$", c["text"].strip())]
print(f"Фіксованих фрагментів, що закінчуються не на межі речення: {len(bad)} з {len(F)}")
for c in bad[:3]: print("  " + show(c))
nxt = F[F.index(bad[0]) + 1]
print(f"\n  початок наступного [{nxt['id']}]: {nxt['text'][:160]}…")
mixed = [c for c in F if len({s for s in re.findall(r'##', c['text'])}) or c["section"] != ""]
cross = [c for c in F if c["doc"] == "KB-01"]
print(f"\nKB-01: фіксованих фрагментів {len(cross)}, структурних {sum(c['doc'] == 'KB-01' for c in S)}; "
      "фіксований фрагмент змішує кілька розділів (метадані section — лише перший):")
print("  " + cross[0]["text"][:400] + "…")
print("\nСтруктурні фрагменти KB-01:"); [print("  " + show(c, 90)) for c in S if c["doc"] == "KB-01"]

Фіксованих фрагментів, що закінчуються не на межі речення: 1 з 26
  [KB-03:f0] стор.1 «Симптоми» (399 ток.): …мої або більшої потужності (для ПК аудиторії — не менше 500 Вт). Якщо з іншим блоком ПК працює стабільно, несправність підтверджено. Розбирати та ремонтувати БЖ

  початок наступного [KB-03:f1]: Найнадійніший метод — тимчасово підключити завідомо справний БЖ такої самої або більшої потужності (для ПК аудиторії — не менше 500 Вт). Якщо з іншим блоком ПК …

KB-01: фіксованих фрагментів 2, структурних 4; фіксований фрагмент змішує кілька розділів (метадані section — лише перший):
  Перегрів процесора — найчастіша причина раптових вимкнень настільних ПК у навчальних аудиторіях. Типові ознаки: комп'ютер вимикається або перезавантажується під навантаженням (гра, рендеринг, компіляція), вентилятор процесорного кулера працює на максимальних обертах і гучно шумить, корпус помітно гарячий, продуктивність різко падає через троттлінг. У журналі подій Windows після такого вимкнення з'…

Стр

[None, None, None, None]

In [4]:
# %% [Р2.1] Embeddings, індекс FAISS (IndexFlatIP) і збереження на диск
t = time.perf_counter()
dense_S = DenseIndex(S); dense_F = DenseIndex(F)
print(f"Модель: {rag.EMB_MODEL}; розмірність {dense_S.vecs.shape[1]}; макс. довжина входу "
      f"{rag.model().max_seq_length} токенів; час кодування {time.perf_counter() - t:.1f} с")
print("Норми векторів (мають бути 1):", np.round(np.linalg.norm(dense_S.vecs, axis=1)[:5], 4))
dense_S.save("index/structural"); dense_F.save("index/fixed")
loaded = DenseIndex.load("index/structural")
print("Збережено:", sorted(str(p) for p in Path("index").rglob("*.*")))
print("Перевірка завантаження з диска: векторів", loaded.index.ntotal, "| той самий результат:",
      [c["id"] for c, _ in loaded.search("TjMax", 3)] == [c["id"] for c, _ in dense_S.search("TjMax", 3)])

Модель: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2; розмірність 384; макс. довжина входу 128 токенів; час кодування 4.0 с
Норми векторів (мають бути 1): [1. 1. 1. 1. 1.]
Збережено: ['index\\chunks_fixed.jsonl', 'index\\chunks_structural.jsonl', 'index\\fixed\\chunks.jsonl', 'index\\fixed\\index.faiss', 'index\\structural\\chunks.jsonl', 'index\\structural\\index.faiss']
Перевірка завантаження з диска: векторів 69 | той самий результат: True


In [5]:
# %% [Р2.2] Пошук top-k з фільтрацією за метаданими (мова, документ)
q = "критична температура процесора"
print("Без фільтра:", [(c["id"], c["lang"], round(s, 3)) for c, s in dense_S.search(q, 5)])
print("lang = en:  ", [(c["id"], c["lang"], round(s, 3)) for c, s in dense_S.search(q, 5, where={"lang": "en"})])
print("doc ∈ {KB-03, KB-17}:", [(c["id"], round(s, 3)) for c, s in
      dense_S.search("чи можна відкривати блок живлення", 3, where={"doc": ["KB-03", "KB-17"]})])

Без фільтра: [('KB-01:s1', 'uk', 0.812), ('KB-21:s0', 'en', 0.738), ('KB-21:s1', 'en', 0.695), ('KB-01:s3', 'uk', 0.585), ('KB-20:s0', 'uk', 0.55)]
lang = en:   [('KB-21:s0', 'en', 0.738), ('KB-21:s1', 'en', 0.695), ('KB-22:s0', 'en', 0.546), ('KB-22:s1', 'en', 0.272)]
doc ∈ {KB-03, KB-17}: [('KB-03:s1', 0.513), ('KB-17:s1', 0.497), ('KB-17:s0', 0.483)]


In [6]:
# %% [Р2.3] Перевірка якості на 5 запитах: чи є релевантний фрагмент у top-3
CHECK = [("Яка допустима напруга на лінії +12 В?", "KB-03:s2"),
         ("Що означає безперервний довгий писк при увімкненні?", "KB-07:s1"),
         ("як відновити завантажувач bootrec", "KB-11:s2"),
         ("скільки триває гарантія на комп'ютери аудиторій", "KB-18:s0"),
         ("комп став тихим, а потім вимкнувся", "KB-02:s0")]
for q, gold in CHECK:
    hits = dense_S.search(q, 3)
    ids = [c["id"] for c, _ in hits]
    print(f"{'✔' if gold in ids else '✘'} {q}\n    очікується {gold}; top-3: {[(i, round(s, 2)) for (c, s), i in zip(hits, ids)]}")

✔ Яка допустима напруга на лінії +12 В?
    очікується KB-03:s2; top-3: [('KB-03:s2', 0.62), ('KB-21:s0', 0.34), ('KB-17:s0', 0.31)]
✔ Що означає безперервний довгий писк при увімкненні?
    очікується KB-07:s1; top-3: [('KB-01:s0', 0.51), ('KB-07:s1', 0.45), ('KB-02:s0', 0.44)]
✔ як відновити завантажувач bootrec
    очікується KB-11:s2; top-3: [('KB-11:s2', 0.73), ('KB-11:s0', 0.55), ('KB-11:s1', 0.53)]
✔ скільки триває гарантія на комп'ютери аудиторій
    очікується KB-18:s0; top-3: [('KB-18:s0', 0.68), ('KB-16:s1', 0.62), ('KB-03:s3', 0.6)]


✔ комп став тихим, а потім вимкнувся
    очікується KB-02:s0; top-3: [('KB-19:s2', 0.44), ('KB-03:s0', 0.36), ('KB-02:s0', 0.35)]


In [7]:
# %% [Р3.1] RAG-конвеєр з цитуванням і відмовою на 10 питаннях
print(rag.RAG_SYSTEM); print()
RAGQ = ["Яка допустима напруга на лінії +12 В за стандартом ATX?",
        "Що означає синій екран з кодом 0x0000007B?",
        "Як перевірити батарейку CMOS і коли її міняти?",
        "Скільки проходів MemTest86 потрібно і що означає одна помилка?",
        "Який строк вирішення заявки з пріоритетом P1?",
        "Чи можна прибирати пил всередині ПК пилососом?",
        "Як відновити завантажувач, якщо пише BOOTMGR is missing?",
        "Які ознаки прихованого майнера?",
        "Скільки коштує заміна відеокарти в сервісному центрі?",            # немає в корпусі
        "Як налаштувати мережевий принтер HP у аудиторії?"]                  # немає в корпусі
H = HybridRetriever(S, dense=dense_S)
by_id = {c["id"]: c for c in S}
ANS = []
for q in RAGQ:
    a = rag.answer(q, H, k=4); a["check"] = rag.check_citations(a, by_id); ANS.append(a)
    print(f"Q: {q}\nA: {a['answer']}\n   джерела top-4: {a['hits']} | цитати: {a['citations']} | "
          f"перевірка: {[(c['id'], c['overlap'], c['cos'], '✔' if c['ok'] else '✘') for c in a['check']]}\n")

Ти — асистент служби ІТ-підтримки. Відповідай українською ЛИШЕ на основі наведених фрагментів бази знань.
Правила:
1. Кожне твердження супроводжуй посиланням на джерело у квадратних дужках, напр. [KB-05:s1].
2. Використовуй лише ідентифікатори фрагментів, наведені нижче; не вигадуй фактів.
3. Якщо у фрагментах немає відповіді — відповідай рівно: «Недостатньо даних».
4. Відповідь — 1–4 речення, без вступу.



Q: Яка допустима напруга на лінії +12 В за стандартом ATX?
A: Допустимі відхилення напруг за стандартом ATX становлять ±5 %: лінія +12 В — від 11,40 до 12,60 В. [KB-03:s2]
   джерела top-4: ['KB-03:s2', 'KB-06:s0', 'KB-17:s0', 'KB-18:s0'] | цитати: ['KB-03:s2'] | перевірка: [('KB-03:s2', 1.0, 0.87, '✔')]

Q: Що означає синій екран з кодом 0x0000007B?
A: Синій екран з кодом 0x0000007B означає зупинку системи через помилку в драйвері відеокарти (VIDEO_TDR_FAILURE) [KB-14:s0].
   джерела top-4: ['KB-08:s0', 'KB-13:s0', 'KB-14:s0', 'KB-19:s1'] | цитати: ['KB-14:s0'] | перевірка: [('KB-14:s0', 0.5, 0.65, '✔')]



Q: Як перевірити батарейку CMOS і коли її міняти?
A: Виміряйте напругу батарейки мультиметром: значення нижче 2,8 В означає, що батарейку слід замінити [KB-04:s2].
   джерела top-4: ['KB-04:s2', 'KB-04:s0', 'KB-11:s1', 'KB-04:s1'] | цитати: ['KB-04:s2'] | перевірка: [('KB-04:s2', 1.0, 0.83, '✔')]



Q: Скільки проходів MemTest86 потрібно і що означає одна помилка?
A: Скільки проходів MemTest86 потрібно: щонайменше 4 повні проходи. [KB-05:s1]

Означає одна помилка: що пам'ять працює нестабільно. [KB-05:s1]
   джерела top-4: ['KB-05:s1', 'KB-06:s2', 'KB-08:s1.1', 'KB-07:s1'] | цитати: ['KB-05:s1'] | перевірка: [('KB-05:s1', 0.71, 0.44, '✔'), ('KB-05:s1', 1.0, 0.58, '✔')]



Q: Який строк вирішення заявки з пріоритетом P1?
A: Вирішення — 4 години. [KB-16:s1]
   джерела top-4: ['KB-16:s1', 'KB-16:s2', 'KB-16:s0', 'KB-18:s0'] | цитати: ['KB-16:s1'] | перевірка: [('KB-16:s1', 1.0, 0.6, '✔')]



Q: Чи можна прибирати пил всередині ПК пилососом?
A: Недостатньо даних. [KB-02:s1]
   джерела top-4: ['KB-05:s3', 'KB-03:s3', 'KB-20:s2', 'KB-02:s1'] | цитати: ['KB-02:s1'] | перевірка: [('KB-02:s1', 0.0, -0.03, '✘')]

Q: Як відновити завантажувач, якщо пише BOOTMGR is missing?
A: Для відновлення завантажувача Windows, якщо пише BOOTMGR is missing, спочатку перевірте порядок завантаження в BIOS, щоб убезпечити, що Windows Boot Manager є першим пристроєм завантаження. Якщо порядок завантаження скидається, то скинути CMOS (KB-04) і перевірте порядок завантаження знову. Якщо це не допоможе, то спробуйте відновити завантажувач за допомогою media creation tool (KB-10).
   джерела top-4: ['KB-11:s0', 'KB-09:s2', 'KB-11:s1', 'KB-08:s1.0'] | цитати: [] | перевірка: []



Q: Які ознаки прихованого майнера?
A: Ознаки прихованого майнера: у Диспетчері задач невідомий процес постійно навантажує процесор або відеокарту на 80–100 % навіть у простої, мережевий трафік зростає без дій користувача. [KB-15:s0]
   джерела top-4: ['KB-15:s0', 'KB-19:s2', 'KB-14:s0', 'KB-03:s0'] | цитати: ['KB-15:s0'] | перевірка: [('KB-15:s0', 1.0, 0.72, '✔')]

Q: Скільки коштує заміна відеокарти в сервісному центрі?
A: Недостатньо даних. [KB-14:s2]
   джерела top-4: ['KB-14:s2', 'KB-02:s2', 'KB-10:s1', 'KB-14:s1'] | цитати: ['KB-14:s2'] | перевірка: [('KB-14:s2', 0.0, 0.16, '✘')]

Q: Як налаштувати мережевий принтер HP у аудиторії?
A: Недостатньо даних.
   джерела top-4: ['KB-20:s2', 'KB-03:s3', 'KB-01:s3', 'KB-10:s1'] | цитати: [] | перевірка: []



In [8]:
# %% [Р3.2] Підсумок автоматичної перевірки цитат
rows = []
for a in ANS:
    rows.append({"питання": a["question"][:48], "відмова": a["refused"], "цитат": len(a["citations"]),
                 "цитати ⊆ top-k": set(a["citations"]) <= set(a["hits"]),
                 "підтверджено": sum(c["ok"] for c in a["check"]), "перевірено": len(a["check"]),
                 "ток. вх/вих": f"{a['tokens_in']}/{a['tokens_out']}", "с": round(a["latency_s"], 1)})
R3 = pd.DataFrame(rows); print(R3.to_string(index=False))
allc = [c for a in ANS for c in a["check"]]
print(f"\nЦитат підтверджено: {sum(c['ok'] for c in allc)}/{len(allc)}; "
      f"відмов на 2 питання без відповіді: {sum(a['refused'] for a in ANS[-2:])}/2; "
      f"хибних відмов на 8 питань з відповіддю: {sum(a['refused'] for a in ANS[:8])}/8")
Path("outputs").mkdir(exist_ok=True)
json.dump(ANS, open("outputs/rag_answers.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

                                         питання  відмова  цитат  цитати ⊆ top-k  підтверджено  перевірено ток. вх/вих    с
Яка допустима напруга на лінії +12 В за стандарт    False      1            True             1           1      977/48 17.9
      Що означає синій екран з кодом 0x0000007B?    False      1            True             1           1      843/49 17.8
  Як перевірити батарейку CMOS і коли її міняти?    False      1            True             1           1      765/45 15.7
Скільки проходів MemTest86 потрібно і що означає    False      1            True             2           2      842/57 19.9
   Який строк вирішення заявки з пріоритетом P1?    False      1            True             1           1      963/18 16.1
  Чи можна прибирати пил всередині ПК пилососом?     True      1            True             0           1      842/14 10.6
Як відновити завантажувач, якщо пише BOOTMGR is     False      0            True             0           0     867/106 23.6
        

In [9]:
# %% [Р4.1] BM25, dense і гібрид RRF: recall@1/3/5 на 26 питаннях
EVAL = [  # (питання, допустимі документи, тип)
 ("Яка допустима напруга на лінії +12 В за стандартом ATX?", {"KB-03"}, "факт"),
 ("Що означає код 0x0000007B?", {"KB-08"}, "код"),
 ("bootrec /fixboot відмовлено в доступі", {"KB-11"}, "код"),
 ("Яка критична температура процесора i5-10400?", {"KB-01", "KB-21"}, "факт"),
 ("Який атрибут SMART показує перепризначені сектори?", {"KB-09"}, "факт"),
 ("Що означають безперервні довгі звукові сигнали при старті?", {"KB-07"}, "факт"),
 ("Скільки проходів MemTest86 рекомендовано?", {"KB-05"}, "факт"),
 ("Чому після вимкнення з розетки збивається годинник?", {"KB-04"}, "синонім"),
 ("комп став тихим а потім гасне через хвилину", {"KB-02"}, "синонім"),
 ("Які строки вирішення заявки з критичним пріоритетом?", {"KB-16"}, "факт"),
 ("Чи можна розкривати БЖ щоб полагодити?", {"KB-03", "KB-17"}, "синонім"),
 ("Як чисто перевстановити драйвер відеокарти?", {"KB-13"}, "факт"),
 ("VIDEO_TDR_FAILURE nvlddmkm.sys", {"KB-08", "KB-13", "KB-14"}, "код"),
 ("DISM /Online /Cleanup-Image /RestoreHealth", {"KB-12"}, "код"),
 ("ознаки прихованого майнера", {"KB-15"}, "факт"),
 ("INC-2026-00417", {"KB-16"}, "код"),
 ("Скільки місяців гарантія на ПК аудиторій?", {"KB-18", "KB-22"}, "факт"),
 ("Монітор пише немає сигналу, що перевірити?", {"KB-19"}, "факт"),
 ("Диск завантажений на 100 % при малому навантаженні", {"KB-20", "KB-09"}, "факт"),
 ("What is the PL2 power limit of the lab processor?", {"KB-21"}, "англ."),
 ("Which graphics card is installed in classroom PCs?", {"KB-22"}, "англ."),
 ("У нас в аудиторії після того як прибиральниця посунула системний блок, комп'ютер при включенні довго пищить і на "
  "екрані нічого немає, що могло статися і що робити?", {"KB-05", "KB-07"}, "довгий"),
 ("Після того як я в біосі ввімкнув профіль пам'яті щоб пришвидшити комп почалися сині екрани а без нього все "
  "стабільно, це поломка планки чи щось інше?", {"KB-06"}, "довгий"),
 ("Чим чистити радіатор і чи підійде звичайний пилосос?", {"KB-17", "KB-01"}, "синонім"),
 ("Чи можна самостійно замінити модуль пам'яті на гарантійному ПК?", {"KB-18"}, "довгий"),
 ("шифрувальник змінив розширення файлів що робити", {"KB-15"}, "синонім"),
]
bm_S = BM25Index(S)
METHODS = {"BM25": lambda q, k: bm_S.search(q, k), "dense": lambda q, k: dense_S.search(q, k),
           "hybrid RRF": lambda q, k: rrf([dense_S.search(q, 20), bm_S.search(q, 20)], k)}
def first_rank(res, gold):
    for i, (c, _) in enumerate(res, 1):
        if c["doc"] in gold: return i
    return None
RANKS = {m: [first_rank(f(q, 5), g) for q, g, _ in EVAL] for m, f in METHODS.items()}
rec = lambda ranks, k: np.mean([r is not None and r <= k for r in ranks])
TAB = pd.DataFrame({m: {f"recall@{k}": round(rec(r, k), 3) for k in (1, 3, 5)} for m, r in RANKS.items()}).T
print(f"Питань: {len(EVAL)}; релевантність — документ з допустимого набору у top-k"); print(TAB.to_string())
types = pd.Series([t for *_, t in EVAL])
print("\nrecall@3 за типом питання:")
print(pd.DataFrame({m: pd.Series([r is not None and r <= 3 for r in RANKS[m]]).groupby(types).mean()
                    for m in METHODS}).round(2).assign(n=types.value_counts()).to_string())
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
Path("figures").mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4))
TAB.T.plot(marker="o", ax=ax, color=["#ED7D31", "#4472C4", "#70AD47"]); ax.set_ylim(0.3, 1.02)
ax.set_title("Recall@k для BM25, dense і гібриду (26 питань)"); ax.grid(alpha=.3); ax.set_ylabel("recall")
fig.tight_layout(); fig.savefig("figures/l11_recall.png", dpi=150); plt.close(fig)

Питань: 26; релевантність — документ з допустимого набору у top-k
            recall@1  recall@3  recall@5
BM25           0.885     0.923     1.000
dense          0.654     0.808     0.923
hybrid RRF     0.808     0.962     0.962

recall@3 за типом питання:
         BM25  dense  hybrid RRF   n
англ.    1.00   0.50        1.00   2
довгий   0.33   0.33        0.67   3
код      1.00   1.00        1.00   5
синонім  1.00   0.60        1.00   5
факт     1.00   1.00        1.00  11


In [10]:
# %% [Р4.2] Де виграє кожен метод: питання з різними рангами
diff = pd.DataFrame({"питання": [q[:60] for q, *_ in EVAL], "тип": types, **{m: RANKS[m] for m in METHODS}})
print(diff[(diff.BM25 != diff.dense) | diff.isna().any(axis=1)].fillna("—").to_string(index=False))

                                                     питання     тип  BM25 dense hybrid RRF
         Чому після вимкнення з розетки збивається годинник? синонім     1   5.0        2.0
                 комп став тихим а потім гасне через хвилину синонім     1   3.0        1.0
                      Чи можна розкривати БЖ щоб полагодити? синонім     1   4.0        1.0
                                  ознаки прихованого майнера    факт     1   3.0        1.0
           What is the PL2 power limit of the lab processor?   англ.     1     —        2.0
          Which graphics card is installed in classroom PCs?   англ.     1   2.0        1.0
У нас в аудиторії після того як прибиральниця посунула систе  довгий     4     —          —
Чи можна самостійно замінити модуль пам'яті на гарантійному   довгий     5   4.0        2.0


In [11]:
# %% [Р5.1] Аналіз помилок: причини промахів (recall@3, гібрид і dense)
def cause(q, gold):
    """Евристична категоризація з ручною перевіркою (див. звіт)."""
    if not any(c["doc"] in gold for c in S): return "немає в корпусі"
    bm_ok = first_rank(bm_S.search(q, 3), gold) is not None
    best = max((c for c in S if c["doc"] in gold), key=lambda c: float(rag.embed([q])[0] @ dense_S.vecs[S.index(c)]))
    if not bm_ok and best["tokens"] > 128: return "chunking (фрагмент довший за вікно моделі)"
    return "embeddings (семантична схожість нижча за інші фрагменти)" if bm_ok else "лексичний розрив (синоніми/розмовна мова)"
miss = []
for m in ("dense", "hybrid RRF"):
    for (q, g, t), r in zip(EVAL, RANKS[m]):
        if r is None or r > 3:
            top = [c["id"] for c, _ in METHODS[m](q, 3)]
            miss.append({"метод": m, "питання": q[:55], "тип": t, "очікувано": ",".join(sorted(g)), "top-3": top,
                         "причина": cause(q, g)})
M = pd.DataFrame(miss); print(M.to_string(index=False))
print("\n", M.groupby(["метод", "причина"]).size().to_string())

     метод                                                 питання     тип   очікувано                            top-3                                                  причина
     dense     Чому після вимкнення з розетки збивається годинник? синонім       KB-04   [KB-01:s2, KB-19:s2, KB-01:s0] embeddings (семантична схожість нижча за інші фрагменти)
     dense                  Чи можна розкривати БЖ щоб полагодити? синонім KB-03,KB-17 [KB-09:s2, KB-08:s1.1, KB-12:s0] embeddings (семантична схожість нижча за інші фрагменти)
     dense       What is the PL2 power limit of the lab processor?   англ.       KB-21   [KB-22:s0, KB-01:s1, KB-06:s0] embeddings (семантична схожість нижча за інші фрагменти)
     dense У нас в аудиторії після того як прибиральниця посунула   довгий KB-05,KB-07   [KB-01:s0, KB-15:s0, KB-03:s3]                лексичний розрив (синоніми/розмовна мова)
     dense Чи можна самостійно замінити модуль пам'яті на гарантій  довгий       KB-18   [KB-03:s3, KB-10:s2, KB-05

In [12]:
# %% [Р5.2] Покращення: переписування запиту (видалення розмовних слів + розширення синонімами з онтології)
from pcdiag.onto import extract_concepts, ALL_CLASSES
FILLER = r"\b(у нас|в аудиторії|після того як|що могло статися|що робити|і що|це|чи щось інше|щоб|звичайний|а потім)\b"
def rewrite(q):
    q2 = re.sub(FILLER, " ", q.lower())
    terms = [ALL_CLASSES.get(c, "") for c in extract_concepts(q)]
    return re.sub(r"\s+", " ", q2 + " " + " ".join(terms)).strip()
for q, *_ in EVAL[21:24]: print(f"«{q[:70]}…»\n   → «{rewrite(q)[:140]}»")
NEW = {m: [first_rank(f(rewrite(q), 5), g) for q, g, _ in EVAL] for m, f in METHODS.items()}
cmp = pd.DataFrame({f"{m} {k}": {"до": round(rec(RANKS[m], int(k[-1])), 3), "після": round(rec(NEW[m], int(k[-1])), 3)}
                    for m in METHODS for k in ("recall@1", "recall@3")}).T
cmp["Δ"] = (cmp["після"] - cmp["до"]).round(3); print(); print(cmp.to_string())
# Альтернатива: фіксовані фрагменти 400 токенів (базова стратегія Р1) — для порівняння конфігурацій
RF = [first_rank(dense_F.search(q, 5), g) for q, g, _ in EVAL]
print(f"\nDense на фіксованих фрагментах 400 ток.: recall@1={rec(RF, 1):.3f}, @3={rec(RF, 3):.3f}, @5={rec(RF, 5):.3f}")

«У нас в аудиторії після того як прибиральниця посунула системний блок,…»
   → «прибиральниця посунула системний блок, комп'ютер при включенні довго пищить і на екрані нічого немає, робити? Сигнали POST»
«Після того як я в біосі ввімкнув профіль пам'яті щоб пришвидшити комп …»
   → «я в біосі ввімкнув профіль пам'яті пришвидшити комп почалися сині екрани а без нього все стабільно, поломка планки ? Оперативна пам'ять»
«Чим чистити радіатор і чи підійде звичайний пилосос?…»
   → «чим чистити радіатор і чи підійде пилосос? Радіатор»



                        до  після      Δ
BM25 recall@1        0.885  0.885  0.000
BM25 recall@3        0.923  0.962  0.039
dense recall@1       0.654  0.577 -0.077
dense recall@3       0.808  0.846  0.038
hybrid RRF recall@1  0.808  0.769 -0.039
hybrid RRF recall@3  0.962  1.000  0.038



Dense на фіксованих фрагментах 400 ток.: recall@1=0.500, @3=0.808, @5=0.846
